## 0 — Setup

In [1]:
import sys
sys.path.insert(0, 'src')

import pathlib
import warnings
import matplotlib
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
%matplotlib inline
matplotlib.rcParams.update({'font.size': 16})

In [ ]:
import spatial_scope
import etl_bgt
import dataset
import evaluation
import modelling

from config import (
    config_download_data,
    config_etl,
    config_modelling,
    config_etl_bgt,
)

## 1 — Download data

Run once to download BGT, municipality boundary and imagery.
Alternatively execute from the terminal:

```bash
python src/download_data.py satellite   # or aerial
```

In [8]:
# Uncomment to run downloads from the notebook
# import download_data
# download_data.main()

## 2 — Spatial scope

In [9]:
rtg = spatial_scope.NlRegionToGeom(**config_download_data['process_bestuurlijkegebieden'])
geom = rtg.from_region_name(config_download_data['region_name'])

In [ ]:
data_array = spatial_scope.build_aerial_mosaic(
    config_etl['postgres']['dir_imagery'],
    config_etl['postgres']['file_format_imagery'],
)

tbbox = spatial_scope.TiffBasedTiledBbox(
    geom,
    rtg.crs,
    config_etl['postgres']['dir_imagery'],
    config_etl['postgres']['file_format_imagery'],
    config_etl['postgres']['tile_width_in_pixels'],
    data_array=data_array,
)

## 3 — BGT ETL

Downloads and transforms BGT ground-truth geometries.
Pass `overwrite=True` to force a re-download.

In [ ]:
gdf = etl_bgt.process(
    geo_filter=tbbox.get_wkt_bbox_with_buffer(),
    crs=tbbox.crs,
    config=config_etl_bgt,
)
gdf.head()

## 4 — Dataset creation

In [ ]:
dsc = dataset.DataSetCreator(
    tbbox, gdf,
    in_storage_dir=config_etl['postgres']['dir_imagery'],
    in_file_name=config_etl['postgres']['file_format_imagery'],
    out_storage_dir=config_etl['dataset']['storage_dir'],
    out_file_name=config_etl['dataset']['agg_tile_data_file_name'],
    featuretypes=config_etl['postgres']['bgt_used_layers'],
)
df = dsc.create()
df.head()

## 5 — Exploratory data analysis

In [ ]:
eda = dataset.ExploratoryDataAnalysis(
    storage_dir=config_etl['dataset']['storage_dir'],
    in_file_name=config_etl['dataset']['agg_tile_data_file_name'],
    tbbox=tbbox,
    sat_image=dsc.data_array,
    out_file_name=config_etl['dataset']['usable_tiles_data_file'],
)

In [ ]:
eda.satellite_and_tile_aggregate_side_by_side(column_name='contrast')

In [ ]:
eda.satellite_and_tile_aggregate_side_by_side(column_name='entropy')

In [ ]:
eda.four_mask_options()

In [ ]:
eda.set_mask(contrast_thld=0.98, entropy_thld=None)
# eda.masked_satellite_image()

In [ ]:
eda.entropy_histogram(write_to_file=True)

In [ ]:
eda.average_label_share(write_to_file=True)

In [ ]:
eda.pixel_count_bar(write_to_file=True)

In [ ]:
eda.export_unmasked_tiles()

## 6 — Data set splitting

In [ ]:
dss = dataset.DataSetSplitter(
    fractions=config_etl['dataset']['split_fractions'],
    stratification_columns=config_etl['dataset']['stratification_columns'],
    bins=config_etl['dataset']['stratification_bins'],
    storage_dir=config_etl['dataset']['storage_dir'],
    agg_tile_data_file_name=config_etl['dataset']['agg_tile_data_file_name'],
    usable_tiles_file_name=config_etl['dataset']['usable_tiles_data_file'],
    data_split_file_name=config_etl['dataset']['data_split_file_name'],
)

In [ ]:
dss.find_best_split(range(100), write_to_file=True)

In [ ]:
dss.plot_distribution(write_to_file=True)

In [ ]:
dss.print_split_table()

In [ ]:
dss.write_data_split_to_file()

## 7 — Training

Sets `modelling.PARAMS` directly from `src/config.py` (bypassing argparse),
then runs the three-phase progressive unfreezing training loop.

In [ ]:
modelling.PARAMS = config_modelling
conn, cur = modelling.get_connection_and_cursor(config_modelling['postgres']['db_name'])
filename_model_parameters = modelling.phased_model_training(cur)

In [ ]:
modelling.write_predictions(conn, cur, filename_model_parameters)
conn.close()

## 8 — Evaluation

Update `log_files` to match the filenames produced by the training run above.

In [ ]:
log_files = [
    # e.g. '20230405094352_phase_1_logfile.json',
    # '20230405112131_phase_2_logfile.json',
    # '20230405125634_phase_3_logfile.json',
]

nbh_cfg = config_download_data['process_wijkenbuurten']
nbh_path = pathlib.Path(nbh_cfg['storage_dir']) / nbh_cfg['file_name']

per_eval = evaluation.PerformanceEvaluation(
    dataset_dir=config_etl['dataset']['storage_dir'],
    datasplit_filename=config_etl['dataset']['data_split_file_name'],
    agg_tile_filename=config_etl['dataset']['agg_tile_data_file_name'],
    pred_dir=config_etl['predictions']['storage_dir'],
    agg_pred_filename=config_etl['predictions']['agg_pred_file_name'],
    log_dir=config_modelling['training']['log_storage_dir'],
    log_files=log_files,
    nbh_path=nbh_path,
    nbh_url=nbh_cfg['wfs_url'],
    tbbox=tbbox,
    geom=geom,
)

### Training process

In [ ]:
per_eval.plot_training_process()

### Confusion matrices

In [ ]:
per_eval.pretty_print_confusion_matrices()

In [ ]:
per_eval.pretty_print_confusion_matrices_by_region()

In [ ]:
per_eval.pretty_print_input_output_correlations(split='test')

### Performance by neighbourhood

In [ ]:
per_eval.plot_performance_by_neighbourhood(metric='recall_pervious', write_to_file=False)

In [ ]:
per_eval.plot_performance_by_neighbourhood(metric='recall_impervious', write_to_file=False)

In [ ]:
nbh_cm = per_eval.get_confusion_matrices_per_neighbourhood(dim=1)
nbh_cm.keys()

In [ ]:
per_eval.pretty_print_confusion_matrices_by_neighbourhood('Binnenstad', 'Zuid')

### Example tiles

In [ ]:
gdf = per_eval.join_all()
gdf.info()

In [ ]:
# Find tile ids — adjust filters as needed
mask = (gdf.split == 'test') & (gdf.nbh == 'Binnenstad')
gdf[mask].head()

In [ ]:
# per_eval.plot_input_labels_and_predicted_labels(tile_id=5837)

In [ ]:
# Tiles dominated by unknown label
mask = (gdf.split == 'test') & (gdf.unknown > 50_000)
gdf[mask].head()